# Integrated Gradients - Vertical analysis of model v2.4

<div class="alert alert-block alert-danger">
<b>WARNING: Running the following cells will lead to severe data loss</b>. The datasets used for this analysis are protected property of the KB National Library in the Netherlands. Therefore, paths to the user's annotations of the application are provided if one would like to test the analysis by themselves. 

However, <b> if you are interested in a deep dive of the XAI analysis of this project, do NOT run the following cells. </b>
</div>

![](img/confusion_matrix.v2.4_optimal_weights.png)

In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers_interpret import SequenceClassificationExplainer
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch

C:\Users\anouk.xs4all\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_v2_4_name = "./models/v2_4_optimal_weights_fold2"
model_v2_4 = AutoModelForSequenceClassification.from_pretrained(model_v2_4_name)
model_v2_4.eval()

tokenizer_v2_4 = AutoTokenizer.from_pretrained("roberta-base")

# Make sure to use the correct dataset for each model
# For privacy reasons, I am not allowed to share the original dataset analyzed here.
# This path allows you to use it for your personal annotated websites.
df_23 = pd.read_json(r".\output\websites_annotated.jsonl", lines=True)

label_encoder = LabelEncoder()
df_23['label'] = label_encoder.fit_transform(df_23['period_bucket'])

Loading weights: 100%|████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 10313.36it/s]


In [4]:
# After a few test runs some generators still showed,  but that is probably due to varying notations of e.g. 'FrontPage.Editor'

import sys
sys.path.append("./script_training")
from labeling_function_v7 import SignalExtractor

# Hard-coding the masking, already defined in the model algorithm but just to be sure.
def mask_payload(payload):
    if not isinstance(payload, str):
        return ""

    for signal_name, pattern in SignalExtractor.regex_patterns.items():
        payload = pattern.sub("", payload)
    return payload

In [5]:
# Testing the masking
subset = df_23[df_23['label'] == 0]
subset_reset = subset.reset_index(drop=True)
samples = []
for idx, row in subset[10:15].iterrows():
    subset_reset['payload'] = mask_payload(row['payload'])
    samples.append({
        # 'url': subset_reset.iloc[idx].get('url', 'unknown'),
        # 'payload_preview': subset_reset.iloc[idx]['payload'][:500],
        'idx': idx
    })

print(samples)

[{'idx': 17}, {'idx': 18}, {'idx': 19}, {'idx': 20}, {'idx': 21}]


### 3.1 Targeting the misclassifications cases

In [6]:
from tqdm import tqdm

def missclassifications(model, tokenizer, df, true_label_idx, miss_label_idx, max_search=200):
    id2label = {0: "19971999", 1: "20002002", 2: "20032006", 3: "20072010"}
    
    true_str = id2label.get(true_label_idx)
    miss_str = id2label.get(miss_label_idx)

    # print("Searching for misclassifications...")

    subset = df[df['label'] == true_label_idx]

    count = 0

    subset_reset = subset.reset_index(drop=True)
    samples = []
    for idx, row in tqdm(subset_reset.iterrows(), total=len(subset_reset), desc="Calculating wrong predictions"):
        row['payload'] = mask_payload(row['payload'])
        
        if idx > max_search:
            print("Reached limit without finding more")
            break
            
        inputs = tokenizer(row['payload'], return_tensors="pt", truncation=True, max_length=512).to(model.device)

        # print(f"Test: {row['payload'][:200]}")
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        predicted_idx = outputs.logits.argmax().item()

        # Check if this specific instance hits the error group
        if predicted_idx == miss_label_idx:
            samples.append({
                'url': subset_reset.iloc[idx].get('url', 'unknown'),
                # 'idx': idx,
                'payload': row['payload'],
                'true_label' : true_str,
                'pred_label' : id2label.get(predicted_idx)
            })

            if len(samples) >=3:
                break
    
    return samples

In [148]:
# We will load them again in the case studies below
# conf_97_03 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 0, 2)
# conf_97_07 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 0, 3)
# conf_03_97 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 2, 0)

In [ ]:
def test(subset):
    item = subset[0]
    warc_payload = item['payload'] 
    url = item['url']
    return (warc_payload, 
            url, 
            print("--- Top Positive Tokens ---"), 
            print("=== Top Positive Tokens ==="))

# print(test(conf_97_03))

### 3.2 Visualizations and manual inspections

In [7]:
def visualization(model, tokenizer, subset, item_index=0):
    cls_explainer = SequenceClassificationExplainer(model, tokenizer)
    model.config.id2label = {0: "1997-1999", 1: "2000-2002", 2: "2003-2006", 3: "2007-2010"}

    item = subset[item_index]
    warc_payload = item['payload'][:1000]
    attributions = cls_explainer(warc_payload)

    attr_df = pd.DataFrame(attributions, columns=['Token', 'Score'])

    return (cls_explainer.visualize(true_class=item['true_label']), item['url'], print("=== Top Positive Tokens ==="), 
        print(attr_df.sort_values(by='Score', ascending=False).head(15)),
        print("\n=== Top Negative Tokens ==="),
        print(attr_df.sort_values(by='Score', ascending=True).head(15)))

In [8]:
import gc
import torch

# Put this right before you run the explainer
gc.collect()
torch.cuda.empty_cache()

#### 3.2.1. 1997–1999 $\rightarrow$ 2007–2010

In [1]:
# Load the sample subset (n=3)
conf_97_07 = missclassifications(model_v2_4, tokenizer_v2_4, df_23, 0, 3)

NameError: name 'missclassifications' is not defined

In [141]:
visualization(model_v2_3, tokenizer_v2_3, conf_97_07, item_index=0)

=== Top Positive Tokens ===
      Token     Score
52       Id  0.467880
50      Pro  0.464831
51        g  0.264179
49       ="  0.189251
69   Border  0.184206
257      ms  0.178079
42        >  0.138457
148      ">  0.120353
56    Front  0.108659
254      ><  0.091301
59   Editor  0.083847
105      ">  0.075236
57     Page  0.061972
91     fram  0.059418
82   frames  0.051972

=== Top Negative Tokens ===
         Token     Score
306         FF -0.225333
305       FFFF -0.222364
32          "> -0.216545
67          =" -0.193627
259    igation -0.172487
31          52 -0.104194
256         -- -0.100046
68   Microsoft -0.096594
30          12 -0.068308
81           < -0.061436
102     border -0.056399
33           Ċ -0.052400
233          > -0.047735
309     master -0.039661
251         -- -0.038351


(<IPython.core.display.HTML object>,
 'www.reto2002.home.xs4all.nl',
 None,
 None,
 None,
 None)

In [142]:
visualization(model_v2_3, tokenizer_v2_3, conf_97_07, item_index=1)

=== Top Positive Tokens ===
       Token     Score
202       52  0.670871
201       12  0.553414
26        Id  0.159317
309     rial  0.107738
249        g  0.098122
199  windows  0.091310
197       et  0.081414
214        "  0.073384
125     uses  0.061642
275       FF  0.061316
132       't  0.059608
111        <  0.052208
155        >  0.048629
284        &  0.047371
246        <  0.044064

=== Top Negative Tokens ===
       Token     Score
116        > -0.106954
217    Front -0.101780
211      Pro -0.089831
3          > -0.089347
209     name -0.069814
250    color -0.065523
30     Front -0.063311
281        n -0.058434
130  browser -0.058237
220   Editor -0.055529
171        - -0.055494
254       "> -0.054095
37         Ċ -0.053684
29        =" -0.048599
151       et -0.047741


(<IPython.core.display.HTML object>,
 'www.wvegmond.home.xs4all.nl',
 None,
 None,
 None,
 None)

In [143]:
visualization(model_v2_3, tokenizer_v2_3, conf_97_07, item_index=2)

=== Top Positive Tokens ===
      Token     Score
58       Id  0.565538
57        g  0.338959
56      Pro  0.334374
59        "  0.173892
55       ="  0.164105
152      ><  0.129354
62    Front  0.115109
54     name  0.087944
123   color  0.074051
89       --  0.060121
137      80  0.057217
245  border  0.056704
65   Editor  0.055053
42        -  0.054451
337       <  0.051402

=== Top Negative Tokens ===
         Token     Score
81           > -0.367404
79          </ -0.173147
122          g -0.100433
118          > -0.097331
144         FF -0.077788
105  Microsoft -0.076008
104         =" -0.075752
127         CC -0.073434
108    content -0.070343
99           > -0.068932
61          =" -0.065448
148         ms -0.064160
142         FF -0.060742
84       style -0.057633
119          < -0.056208


(<IPython.core.display.HTML object>,
 'www.hrensink.home.xs4all.nl',
 None,
 None,
 None,
 None)

#### 3.2.3. 2003–2006 $\rightarrow$ 1997–1999
Why are mid-2000s pages leaking into the 90s bucket?

In [144]:
# Load the sample subset (n=3)
conf_03_97 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 2, 0)

Calculating wrong predictions:  11%|█████▋                                            | 14/123 [00:15<01:58,  1.08s/it]


In [145]:
visualization(model_v2_3, tokenizer_v2_3, conf_03_97, item_index=0)

=== Top Positive Tokens ===
    Token     Score
205    ER  0.387100
169    ER  0.318053
202   IGN  0.213277
135     >  0.209209
10    TIT  0.194301
224     >  0.191650
192    BR  0.190545
11     LE  0.172523
311  IGHT  0.162238
204  CENT  0.159367
187    ID  0.145605
92    the  0.137703
201    AL  0.132135
20    TIT  0.128182
6    HEAD  0.125796

=== Top Negative Tokens ===
    Token     Score
45     95 -0.120313
82      { -0.111918
146     > -0.106413
14      a -0.092640
124   ="# -0.076820
220     > -0.064325
225     Ċ -0.063284
369     w -0.062854
25    ETA -0.061746
189     = -0.060884
13     In -0.058786
185    53 -0.058393
182    HE -0.058013
304     " -0.053862
106     Ċ -0.053729


(<IPython.core.display.HTML object>,
 'www.bk.home.xs4all.nl',
 None,
 None,
 None,
 None)

In [146]:
visualization(model_v2_3, tokenizer_v2_3, conf_03_97, item_index=1)

=== Top Positive Tokens ===
         Token     Score
57         ETA  0.305754
11          LE  0.263042
194  Microsoft  0.248850
179  Microsoft  0.220345
21          LE  0.197871
56           M  0.194675
10         TIT  0.188603
22           >  0.187889
6         HEAD  0.183583
20         TIT  0.180064
55           <  0.173408
19          </  0.157416
181          "  0.125461
196          "  0.124191
59          ="  0.114448

=== Top Negative Tokens ===
      Token     Score
256      ms -0.308498
279     100 -0.245169
126       ! -0.146831
186      01 -0.113661
140     com -0.083294
195  Border -0.063449
66      ier -0.063005
133    http -0.060277
262  border -0.059196
134     :// -0.056606
141       / -0.052351
222      pg -0.045621
232    FFFF -0.044539
96       en -0.043766
200      "> -0.042113


(<IPython.core.display.HTML object>,
 'www.ahcb.home.xs4all.nl',
 None,
 None,
 None,
 None)

In [147]:
visualization(model_v2_3, tokenizer_v2_3, conf_03_97, item_index=2)

=== Top Positive Tokens ===
    Token     Score
265    ID  0.396794
266    TH  0.370483
125   Kam  0.292665
300   AME  0.197916
301     B  0.182766
241   ORD  0.169901
207    </  0.160032
114  Pier  0.142194
213  HEAD  0.131097
302   ORD  0.128784
290    ES  0.116816
209    LE  0.103534
242    ER  0.099844
240     B  0.097884
217  AMES  0.094013

=== Top Negative Tokens ===
    Token     Score
270   MAR -0.235236
308   ORD -0.126670
126   ers -0.107751
341  IGHT -0.097002
268     0 -0.090513
271     G -0.083456
127     , -0.073728
348  AMES -0.073132
101    =" -0.071513
235   ORD -0.063472
309    ER -0.061881
236    ER -0.055933
350     R -0.046917
298     " -0.045241
259     0 -0.043873


(<IPython.core.display.HTML object>,
 'www.boule1.home.xs4all.nl',
 None,
 None,
 None,
 None)

### 3.3 Correct classifications

#### 3.3.1 2007–2010 $\leftrightarrow$ 2007-2010
What constitutes a correct 2007-2010 prediction?

In [151]:
conf_07_07 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 3, 3)

Calculating wrong predictions:   3%|█▋                                                  | 3/92 [00:03<01:56,  1.31s/it]


In [152]:
visualization(model_v2_3, tokenizer_v2_3, conf_07_07, item_index=0)

=== Top Positive Tokens ===
       Token     Score
144      css  0.472106
69        Id  0.317359
67       Pro  0.259878
254        !  0.254467
66        ="  0.135876
68         g  0.129045
143        /  0.116009
255       --  0.099221
51         Ċ  0.077884
65      name  0.075599
74      Page  0.063483
73     Front  0.057967
146     href  0.053532
145        "  0.052465
258  igation  0.047595

=== Top Negative Tokens ===
     Token     Score
48      12 -0.468956
256     ms -0.306668
49      52 -0.165485
47       - -0.159384
50      "> -0.125668
43   chars -0.096795
315     MS -0.075283
72      =" -0.071627
123      > -0.069026
44      et -0.062829
316      F -0.052775
314      Ċ -0.045858
6        > -0.038435
70       " -0.035492
59       > -0.034783


(<IPython.core.display.HTML object>,
 'www.oene.home.xs4all.nl',
 None,
 None,
 None,
 None)